# Natural language to SQL





## Configure Hugging Face token

To run this lab, first you'll need to get a Hugging Face token. Follow these steps:

1. Create a free account at [https://huggingface.co](https://huggingface.co/)
2. Go to Settings → Access Tokens
3. Generate a token (read access is enough)

Then, you'll need to upload this notebook to Google Colab and configure the token you've just created:

1. Go to [https://colab.research.google.com](https://colab.research.google.com)
2. Upload this notebook (e.g., File → Upload notebook)
3. Click on "🔑 Secrets" (left panel)
4. Select "Add new secret":
    - Name: HF_TOKEN
    - Value: paste your Hugging Face token

Note:
- Run this notebook directly in the Google Colab browser interface (https://colab.research.google.com/).
- Other environments (such as VS Code with the Google Colab extension) may not support Colab Secrets, which means your HF_TOKEN may not be accessible and the notebook may fail to run as expected.


In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
# Check if Hugging Face token is loaded correctly

try:
    token = userdata.get("HF_TOKEN")

    if token:
        print("HF_TOKEN found. You're ready to continue! ✅")
    else:
        print(
            "HF_TOKEN is not available ❌ \n"
            "Please follow the setup instructions above to add your Hugging Face "
            "token as a Colab Secret named 'HF_TOKEN'."
        )
except Exception:
    print(
        "Unable to access 'HF_TOKEN' ❌ \n"
        "Make sure you're running this notebook in the Google Colab web interface "
        "and that you've added a Colab Secret named 'HF_TOKEN'."
    )

## Initial Setup

In [ ]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [ ]:
model_name = "defog/sqlcoder-7b"

`defog/sqlcoder-7b` is based on the Mistral 7B architecture and has been fine-tuned for text-to-SQL tasks, enabling it to generate high-quality SQL queries from natural language.


We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [ ]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [ ]:
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map='auto',
                    use_cache = True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

This function wraps the call to *model.generate*

In [ ]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [ ]:
sp_nl2sql = """### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- Remember that revenue is price multiplied by quantity
- Remember that cost is supply_price multiplied by quantity
- Remember that profit is calculated as: (products.price - product_suppliers.supply_price) * sales.quantity
- To find the most profitable product, sum the profit grouped by the product, order the results in descending order, and limit the output to 1 result.

### Input
Generate a SQL query that answers the question below.
This query will run on a database whose schema is represented in this string:

CREATE TABLE products (
  product_id INTEGER PRIMARY KEY, -- Unique ID for each product
  name VARCHAR(50), -- Name of the product
  price DECIMAL(10,2), -- Price of each unit of the product
  quantity INTEGER  -- Current quantity in stock
);

CREATE TABLE customers (
   customer_id INTEGER PRIMARY KEY, -- Unique ID for each customer
   name VARCHAR(50), -- Name of the customer
   address VARCHAR(100) -- Mailing address of the customer
);

CREATE TABLE salespeople (
  salesperson_id INTEGER PRIMARY KEY, -- Unique ID for each salesperson
  name VARCHAR(50), -- Name of the salesperson
  region VARCHAR(50) -- Geographic sales region
);

CREATE TABLE sales (
  sale_id INTEGER PRIMARY KEY, -- Unique ID for each sale
  product_id INTEGER, -- ID of product sold
  customer_id INTEGER,  -- ID of customer who made purchase
  salesperson_id INTEGER, -- ID of salesperson who made the sale
  sale_date DATE, -- Date the sale occurred
  quantity INTEGER -- Quantity of product sold
);

CREATE TABLE product_suppliers (
  supplier_id INTEGER PRIMARY KEY, -- Unique ID for each supplier
  product_id INTEGER, -- Product ID supplied
  supply_price DECIMAL(10,2) -- Unit price charged by supplier
);

-- sales.product_id can be joined with products.product_id
-- sales.customer_id can be joined with customers.customer_id
-- sales.salesperson_id can be joined with salespeople.salesperson_id
-- product_suppliers.product_id can be joined with products.product_id

### Response
Based on your instructions, here is the SQL query I have generated to answer the question
`{question}`:
```sql
"""

In [ ]:
sp_nl2sql = sp_nl2sql.format(question="What is the most profitable product?")
print(sp_nl2sql)

In [ ]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql")[-1].split("```")[0].split(";")[0].strip() + ";")

The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [ ]:
sp_nl2sql2 = """### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the database structure**

### Input
Generate a SQL query that answers the question below.
This query will run on a database whose schema is represented in this string:

CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  name VARCHAR(50),
  price DECIMAL(10,2),
  quantity INTEGER
);

CREATE TABLE customers (
   customer_id INTEGER PRIMARY KEY,
   name VARCHAR(50),
   address VARCHAR(100)
);

CREATE TABLE salespeople (
  salesperson_id INTEGER PRIMARY KEY,
  name VARCHAR(50),
  region VARCHAR(50)
);

CREATE TABLE sales (
  sale_id INTEGER PRIMARY KEY,
  product_id INTEGER,
  customer_id INTEGER,
  salesperson_id INTEGER,
  sale_date DATE,
  quantity INTEGER
);

CREATE TABLE product_suppliers (
  supplier_id INTEGER PRIMARY KEY,
  product_id INTEGER,
  supply_price DECIMAL(10,2)
);

### Samples & Response
Here are examples of questions and their corresponding SQL queries:

Question: "Show all customer names and their addresses."
SQL: SELECT name, address FROM customers;

Question: "Calculate the total revenue generated from all sales."
SQL: SELECT SUM(p.price * s.quantity) AS total_revenue FROM sales s JOIN products p ON s.product_id = p.product_id;

Based on your instructions, here is the SQL query I have generated to answer the question
`{question}`:
```sql
"""

In [ ]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))

In [ ]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql")[-1].split("```")[0].split(";")[0].strip() + ";")

The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [ ]:
sp_nl2sql3b = """### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the database structure**

### Input
Generate a SQL query that answers the question below.
This query will run on a database whose schema is represented in this string:

CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  name VARCHAR(50),
  price DECIMAL(10,2),
  quantity INTEGER
);

CREATE TABLE customers (
   customer_id INTEGER PRIMARY KEY,
   name VARCHAR(50),
   address VARCHAR(100)
);

CREATE TABLE salespeople (
  salesperson_id INTEGER PRIMARY KEY,
  name VARCHAR(50),
  region VARCHAR(50)
);

CREATE TABLE sales (
  sale_id INTEGER PRIMARY KEY,
  product_id INTEGER,
  customer_id INTEGER,
  salesperson_id INTEGER,
  sale_date DATE,
  quantity INTEGER
);

CREATE TABLE product_suppliers (
  supplier_id INTEGER PRIMARY KEY,
  product_id INTEGER,
  supply_price DECIMAL(10,2)
);

### Samples
Question: "Show all customer names and their addresses."
SQL: SELECT name, address FROM customers;

Question: "Calculate the total revenue generated from all sales."
SQL: SELECT SUM(p.price * s.quantity) AS total_revenue FROM sales s JOIN products p ON s.product_id = p.product_id;

### Response
Based on your instructions, here is the SQL query I have generated to answer the question
`{question}`:
```sql
"""

In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql")[-1].split("```")[0].split(";")[0].strip() + ";")

#Now the question in spanish.


In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Devuelve el nombre del empleado mejor pagado.")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql")[-1].split("```")[0].split(";")[0].strip() + ";")

The generated SQL command is the same regardless of where we have placed the examples.

#Conclusions.

Let's see the three SQL's together.

* SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.ID_Usr = salary.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* Spanish Question: SELECT e.name
     FROM employees e
     JOIN salary s ON e.ID_Usr = s.ID_Usr
     WHERE s.salary = (SELECT MAX(salary) FROM salary)
     GROUP BY e.name
     ORDER BY COUNT(studies.ID_study) DESC
     LIMIT 1;


**The model has demonstrated that it is highly efficient in crafting SQL.** Additionally, it pays a lot of attention, perhaps too much, to the examples we provide. Clearly, these examples should be crafted by one of the best SQL programmers we have access to, though their use may not be essential.

On the other hand, although the model is clearly very proficient in SQL generation, during the creation of the notebook, I have encountered several issues because the commands need to be extremely clear. It doesn't handle typos well (which should not exist).

It appears to have some issues when it receives commands in Spanish. I assume this problem would be present in any language other than English. Therefore, since it's a tool that could be used by non-technical personnel, this should be considered in environments where English is not the primary language.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

# 📄 Report: Text-to-SQL Prompt Engineering Evaluation

## 1. Introduction

This report evaluates the performance and robustness of an open-source Text-to-SQL model (`sqlcoder-7b`) across several prompt engineering variations.

Out of curiosity, the database schema from the previous task—containing the tables **products**, **customers**, **salespeople**, **sales**, and **product_suppliers**—was intentionally reused for this experiment. This created a deliberate mismatch with the target question:

> **"Return the name of the best paid employee."**

The mismatch served as a **stress test** for the model's contextual awareness, reasoning capabilities, and error handling when the required information was not available in the database schema.

---

# 2. Methodology & Prompt Variations

## Variation 1 – Baseline

- Included strict operational rules.
- Explicitly defined mathematical formulas for:
  - Revenue
  - Cost
  - Profit

## Variations 2 & 3 – Few-Shot Prompting

- Added several natural language examples.
- Placed examples inside a dedicated `### Samples` section.
- Intended to guide the model toward the desired SQL structure and JOIN patterns.

## Special Test Cases

The model was additionally evaluated using:

- 🇬🇧 **English:** *"Return the name of the best paid employee."*
- 🇪🇸 **Spanish:** *"Devuelve el nombre del empleado mejor pagado."*

The objective was to evaluate the model's cross-lingual robustness while operating under the same schema constraints.

---

# 3. Empirical Model Outputs

## Variations 1–3 (English Question)

```sql
SELECT salespeople.name
FROM salespeople
JOIN (
    SELECT sales.salesperson_id,
           MAX(sales.sale_id) AS max_sale_id
    FROM sales
    GROUP BY sales.salesperson_id
) AS max_sales
ON salespeople.salesperson_id = max_sales.salesperson_id
WHERE max_sales.max_sale_id = (
    SELECT MAX(max_sale_id)
    FROM (
        SELECT sales.salesperson_id,
               MAX(sales.sale_id) AS max_sale_id
        FROM sales
        GROUP BY sales.salesperson_id
    ) AS max_sales
)
ORDER BY salespeople.name NULLS LAST;
```

---

## Variation 4 (Spanish Question)

```sql
SELECT
    c.name,
    s.sale_date,
    p.price,
    ROW_NUMBER() OVER (
        PARTITION BY c.customer_id
        ORDER BY s.sale_date DESC
    ) AS recent_sale_rank
FROM customers c
JOIN sales s
    ON c.customer_id = s.customer_id
JOIN products p
    ON s.product_id = p.product_id;
```

---

# 4. Findings & Failure Analysis

## Logic Trap & Blind Optimization

Across all three English prompt variations, the model exhibited the **same logical failure**.

Since the database schema contained **no `employees` table** and **no `salary` attribute**, a robust system should have concluded that the requested information was unavailable.

Instead, the model attempted to satisfy the prompt by generating a valid SQL query that selected the salesperson associated with the **maximum `sale_id`**.

### Analysis

The model incorrectly interpreted the sequential primary key (`sale_id`) as an indicator of financial compensation.

As a consequence:

- it produced syntactically valid SQL,
- successfully followed the requested query structure,
- but completely failed semantically.

Notably, changing the prompt design from a **Baseline** prompt to **Few-Shot prompting** had **no measurable effect**. The model consistently prioritized structural compliance over semantic correctness.

---

## Language Barrier & Catastrophic Hallucination

The Spanish prompt resulted in an even more severe failure.

Instead of attempting to answer the original question, the model generated a completely unrelated analytical query using a window function:

```sql
ROW_NUMBER() OVER (...)
```

The generated query ranked each customer's most recent purchase rather than identifying the highest-paid employee.

### Analysis

The model failed to correctly process the Spanish input.

Rather than:

- translating the request,
- recognizing that the schema lacked the necessary information, or
- responding with an uncertainty statement,

it hallucinated an entirely unrelated SQL query involving the **customers**, **sales**, and **products** tables.

This demonstrates that multilingual inputs can significantly increase the likelihood of semantic drift when unsupported by the underlying schema.

---

# 5. Conclusion & Lessons Learned

The experiments highlight two important limitations of current Text-to-SQL models.

## Lack of Schema Validation

The model demonstrated **little semantic validation** of the user's request.

Instead of determining whether the required information actually existed within the database schema, it consistently generated syntactically correct SQL regardless of whether the query was answerable.

This behavior can easily produce convincing but fundamentally incorrect results.

## Limits of Prompt Engineering

Few-shot prompting successfully influenced the **syntactic structure** of the generated SQL.

However, it did **not** improve semantic reasoning when the requested information was absent from the schema.

Consequently, prompt engineering alone cannot compensate for missing domain knowledge or insufficient schema validation. Reliable Text-to-SQL systems therefore require additional mechanisms—such as schema-aware validation or explicit uncertainty handling—to prevent hallucinated SQL generation.